# 17 Normal Additional Candidate Search

Controlled step only: find and RIB-prescreen additional Normal replacement candidates for unresolved TRAIN controls. This notebook does not download UPDATE files, change the 30-minute rule, access Validation/Holdout, train models, or alter frozen roster/partition files.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
from bgp_incident_detection.normal_candidate_search_v3 import run

PILOT = ROOT / 'data' / 'pilot'
unresolved = ['Hijack:58159', 'Hijack:97511', 'Leak:426483']
unresolved

## Execute Search

The search evaluates wider date offsets and requires RIB visibility before a candidate can become `UPDATE_READY`. RouteViews UPDATE URLs are checked with HEAD only.

In [ ]:
summary = run()
summary

## Candidate Outcomes

In [ ]:
additional = pd.read_csv(PILOT / 'normal_additional_replacement_candidates_v1.csv')
manifest = pd.read_csv(PILOT / 'normal_second_pilot_candidate_manifest_v1.csv')
plan = pd.read_csv(PILOT / 'normal_second_pilot_acquisition_plan_v1.csv')

display(additional.groupby(['associated_incident_uid', 'prescreen_status']).size().reset_index(name='count'))
display(manifest.groupby(['associated_incident_uid', 'category', 'candidate_source']).size().reset_index(name='candidate_count'))
display(plan.groupby(['associated_incident_uid', 'category', 'collector', 'remote_available']).size().reset_index(name='update_file_count'))

## Audit Checks

In [ ]:
assert set(additional['associated_incident_uid']) == set(unresolved)
assert (additional['candidate_duration_minutes'] >= 30).all()
assert set(manifest['prescreen_status']) == {'UPDATE_READY'}
assert set(plan['remote_available'].astype(str)) == {'True'}
assert plan['mrt_url'].str.contains('/UPDATES/').all()
assert plan['notes'].str.contains('dry-run only, no UPDATE download').all()
print('PASS: additional Normal candidate search produced dry-run UPDATE-ready candidates without UPDATE downloads.')